# U-Net

**Задача**: сегментация зубов    
**Среда**: Google Colab

Особенности реализации:
- **Архитектура**: классическая U-Net
- **Функция потерь** Combined:
  - CrossEntropy Loss, вес = 1.0
  - Dice Loss, вес = 1.0
  - Focal Loss вес = 0.5
- **Аугментации** есть

**Датасет**: открытый датасет **teeth-seg-3537 Computer Vision Model** [godento2](https://universe.roboflow.com/godento2/teeth-seg-3537-iaky1), источник Roboflow.

## Установка пакетов и импорт библиотек

In [ ]:
# Установка tree
!apt-get install tree -qqq > /dev/null

In [ ]:
import os
from google.colab import drive
import zipfile
from pathlib import Path
import yaml
import json
from tqdm import tqdm
from datetime import datetime
import random
import traceback

import cv2
from PIL import Image
import numpy as np
import argparse

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms

from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, classification_report
from scipy import ndimage

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt
import seaborn as sns
# настройка визуализации
%matplotlib inline
plt.style.use('seaborn-v0_8')
%config InlineBackend.figure_format = "retina"

In [ ]:
# скачиваем свой модуль с необходимыми функциями из своего репозитория
!wget https://raw.githubusercontent.com/drSever/MIPT_X-RayDent/refs/heads/master/01_teeth_segmentation/02_U-Net/functions_unet.py --no-cache

In [ ]:
# импорт из скрипта
import functions_unet as f
from functions_unet import (
    TeethSegmentationDataset,
    get_loss_function,
    SegmentationMetrics,
    load_checkpoint,
    plot_training_history,
    load_model
    )

## Для воспроизводимости экспериментов

In [ ]:
SEED = 42
f.set_seed(SEED)

## Получение датасета (нет детских снимков, формат масок YOLOv8)

In [ ]:
DATASET_PATH_ZIP = '/content/gdrive/MyDrive/Colab_Notebooks/Startup/02_Datasets/Godento2v2.zip'
DATASET_PATH_UNZIP = '/content'
DATASET_DIR = '/content/dataset'

In [ ]:
drive.mount('/content/gdrive')

In [ ]:
# Распаковываем
with zipfile.ZipFile(DATASET_PATH_ZIP, 'r') as zip_ref:
    for file in tqdm(zip_ref.namelist(), desc='Распаковка архива'):
        zip_ref.extract(file, DATASET_PATH_UNZIP)

# переименовываем каталог с распакованным датасетом
os.rename('/content/Godento2v2', '/content/dataset')

print(f"\nАрхив успешно распакован в: {DATASET_DIR}")

In [ ]:
!tree {DATASET_DIR} -L 3 -d  # -d только директории, -L 3 - глубина 3 уровня

## Обучение

### Настройки

In [ ]:
### Настройки и параметры

DATASET_PATH = '/content/dataset'  # Путь к датасету
LEARNING_RATE = 1e-4
BATCH_SIZE = 8 # 4-L4, 8-A100
IMG_SIZE = 512
NUM_CLASSES = 33  # 32 зуба + фон
PATIENCE = 15  # Для early stopping
MIN_LR = 1e-7  # Минимальный learning rate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_PIN_MEMORY = torch.cuda.is_available()  # Используем pin_memory только с CUDA

## Сохранение результатов
RESULTS_PATH = '/content/gdrive/MyDrive/Colab_Notebooks/Startup/04_TeethSegmentation/U-Net/'
SAVE_DIR = RESULTS_PATH + "Simple_3"

In [ ]:
# Функция для определения трансформаций для обучения нашей модели
def get_train_transforms(img_size=IMG_SIZE):
    """
    Аугментации для ортопантомограмм
    Специфично для сегментации зубов на рентгеновских снимках.
    """
    return A.Compose([
        # ------------------------------------------------------------
        # 1. ГЕОМЕТРИЧЕСКИЕ ПРЕОБРАЗОВАНИЯ
        # ------------------------------------------------------------
        A.Affine(
            scale=(0.95, 1.05),               # Масштаб: случайное увеличение/уменьшение до ±5%
            translate_percent=(0.03, 0.03),   # Сдвиг по горизонтали/вертикали до 3% от размера
            rotate=(-3, 3),                   # Поворот до ±3 градусов
            p=0.5,                            # Вероятность применения 50%
            border_mode=cv2.BORDER_CONSTANT,  # Новые области заполняются константой (чёрным)
            fill=0,                           # Значение для заполнения на изображении (0 — чёрный)
            fill_mask=0                       # Значение для заполнения на маске (0 — фон)
        ),
        # ОПИСАНИЕ: лёгкие аффинные искажения имитируют небольшие повороты головы пациента,
        # разный масштаб съёмки и смещения челюсти в кадре. Все изменения незначительны,
        # чтобы не нарушить анатомическую структуру зубного ряда.

        # ------------------------------------------------------------
        # 2. МЯГКИЕ ЭЛАСТИЧНЫЕ ДЕФОРМАЦИИ (применяются редко)
        # ------------------------------------------------------------
        A.ElasticTransform(
            alpha=0.5,                            # Интенсивность сдвига пикселей (малое значение → слабая деформация)
            sigma=25,                             # Степень размытия поля деформации (сглаживает искажения)
            p=0.1,                                # Вероятность 10% — очень осторожное применение
            mask_interpolation=cv2.INTER_NEAREST  # Для масок — интерполяция без сглаживания
        ),
        # ОПИСАНИЕ: ElasticTransform моделирует незначительные неоднородности тканей или
        # естественные искривления челюсти. Параметры подобраны так, чтобы зубы оставались
        # узнаваемыми, но контуры слегка «дышали». Высокая вероятность не используется,
        # чтобы не создавать неестественных форм.

        # ------------------------------------------------------------
        # 3. КОРРЕКЦИЯ КОНТРАСТА И ЯРКОСТИ (РЕНТГЕН-СПЕЦИФИЧНЫЕ)
        # ------------------------------------------------------------
        A.CLAHE(
            clip_limit=2.0,             # Порог ограничения гистограммы (выше — сильнее контраст)
            tile_grid_size=(8, 8),      # Размер ячеек для локального выравнивания на изображении (ОПТГ)
            p=0.5
        ),
        # ОПИСАНИЕ: CLAHE (Contrast Limited Adaptive Histogram Equalization) —
        # стандартный метод для рентгеновских снимков. Улучшает локальный контраст,
        # делая более чёткими границы зубов и корней, особенно на затемнённых участках.
        # Параметры (8,8) и clip_limit=2.0 дают умеренное усиление.

        A.RandomBrightnessContrast(
            brightness_limit=0.08,      # Изменение яркости до ±8%
            contrast_limit=0.08,        # Изменение контраста до ±8%
            p=0.5
        ),
        # Описание: Имитирует вариации экспозиции при съёмке — разные аппараты,
        # настройки яркости, толщина мягких тканей.

        # ------------------------------------------------------------
        # 4. ИМИТАЦИЯ АРТЕФАКТОВ (CoarseDropout) (применяются редко)
        # ------------------------------------------------------------
        A.CoarseDropout(
            num_holes_range=(1, 2),         # Количество прямоугольных артефактов: 1 или 2
            hole_height_range=(0.02, 0.04), # Высота артефакта 2–8% от высоты изображения
            hole_width_range=(0.02, 0.04),  # Ширина артефакта 2–8% от ширины
            fill=0,                         # Заливка артефакта чёрным (0) на изображении
            fill_mask=0,                    # Заливка артефакта фоном (0) на маске
            p=0.05                          # Вероятность 5% — редкое событие
        ),
        A.CoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.02, 0.04),
            hole_width_range=(0.02, 0.04),
            fill=128,                       # Заливка артефакта серым (128) на изображении
            fill_mask=0,
            p=0.05
        ),
        A.CoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.02, 0.04),
            hole_width_range=(0.02, 0.04),
            fill=255,                       # Заливка артефакта белым (255) на изображении
            fill_mask=0,
            p=0.05
        ),
        # ОПИСАНИЕ: CoarseDropout заменяет случайные прямоугольные области на указанный цвет.
        # В контексте ортопантомограмм это имитирует:
        #   - артефакты движения (смазанные участки);
        #   - наложение посторонних предметов (зажимы, маркеры);
        #   - отсутствие части зуба (например, разрушенная коронка);
        #   - переэкспонированные участки, пломбы, коронки, металлические вкладки.
        # Маленькая вероятность и небольшие размеры артефактов (2–8%) предотвращают
        # чрезмерное зашумление датасета и позволяют модели научиться игнорировать
        # локальные дефекты.

        # ------------------------------------------------------------
        # 5. ДОБАВЛЕНИЕ ШУМА (GaussNoise)
        # ------------------------------------------------------------
        A.GaussNoise(std_range=(0.01, 0.04), p=0.2),
        # ОПИСАНИЕ: Добавляет гауссов шум, имитируя зернистость рентгеновской
        # плёнки или электронные шумы. Параметры std_range
        # дают умеренный уровень шума, достаточный для повышения робастности модели,
        # но не разрушающий мелкие детали (например, периодонтальную щель).

        # ------------------------------------------------------------
        # 6. НОРМАЛИЗАЦИЯ И ПРЕОБРАЗОВАНИЕ В ТЕНЗОР PyTorch
        # ------------------------------------------------------------
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2()
    ])

def get_val_transforms(img_size=IMG_SIZE):
    """
    Трансформации для валидационного и тестового наборов
    """
    return A.Compose([
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2()
    ])

### Архитектура U-Net

In [ ]:
class DoubleConv(nn.Module):
    """
    Двойной сверточный блок, применяемый в U-Net.
    Состоит из двух последовательных сверток 3x3, каждая из которых
    сопровождается нормализацией по батчу и активацией ReLU.
    """
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=33, features=[64, 128, 256, 512]):
        """
        U-Net для instance-сегментации зубов на ортопантомограммах
        Имеет классическую U-образную структуру: нисходящий путь, бутылочное горло и восходящий путь
        Skip-connections соединяют соответствующие уровни нисходящего и восходящего путей
        Args:
            in_channels: 1 для grayscale рентгеновских снимков
            out_channels: 33 (32 класса зубов + 1 фон)
            features: количество каналов на каждом уровне
        """
        super(UNet, self).__init__()
        self.ups = nn.ModuleList() # Список для восходящих слоев
        self.downs = nn.ModuleList() # Список для нисходящих слоев
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Создание нисходящего пути (encoder)
        for feature in features:
            # Добавляем двойной сверточный блок для текущего уровня
            self.downs.append(DoubleConv(in_channels, feature))
            # Обновляем in_channels для следующего уровня
            in_channels = feature

        # Bottleneck
        # Удваиваем количество каналов по сравнению с последним уровнем encoder'а
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        # Создание восходящего пути (decoder)
        for feature in reversed(features):
            # Транспонированная свертка для увеличения разрешения в 2 раза
            self.ups.append(
                nn.ConvTranspose2d(
                    feature*2,  # Входные каналы
                    feature,    # Выходные каналы
                    kernel_size=2,
                    stride=2,   # Увеличивает размерность в 2 раза
                )
            )
            # Двойной сверточный блок после объединения с skip-connection
            self.ups.append(DoubleConv(feature*2, feature))

        # Финальная свертка 1x1 для получения нужного количества классов
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        """
        Прямой проход через U-Net.

        Args:
            x (torch.Tensor): Входной тензор формы [batch_size, 1, H, W]

        Returns:
            torch.Tensor: Выходной тензор формы [batch_size, 33, H, W]
        """
        # Список для хранения skip-connections
        skip_connections = []

        # Нисходящий путь
        for down in self.downs:
            # Применяем двойную свертку
            x = down(x)
            # Сохраняем результат для skip-connection
            skip_connections.append(x)
            # Применяем max-pooling для уменьшения разрешения
            x = self.pool(x)

        # Bottleneck (самый низкий уровень)
        x = self.bottleneck(x)

        # Разворачиваем список skip-connections в обратном порядке
        # для соответствия уровням восходящего пути
        skip_connections = skip_connections[::-1]

        # Восходящий путь
        # Идем с шагом 2, т.к. каждый уровень decoder'а состоит из двух слоев:
        # 1) транспонированная свертка, 2) двойная свертка
        for idx in range(0, len(self.ups), 2):
            # Транспонированная свертка для увеличения разрешения
            x = self.ups[idx](x)

            # Получаем соответствующую skip-connection
            skip_connection = skip_connections[idx//2]

            # Проверяем совпадение размерностей (может отличаться из-за округлений)
            if x.shape != skip_connection.shape:
                # Интерполяция для точного совпадения размеров
                x = F.interpolate(x, size=skip_connection.shape[2:], mode='bilinear', align_corners=True)

            # Объединяем по каналам: skip-connection и выход транспонированной свертки
            concat_skip = torch.cat((skip_connection, x), dim=1)

            # Применяем двойную свертку к объединенному тензору
            x = self.ups[idx+1](concat_skip)

        # Финальная свертка для получения карты сегментации
        return self.final_conv(x)

### Функция обучения

In [ ]:
# функция обучения
def train_teeth_segmentation(
    dataset_path=DATASET_PATH,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    num_epochs=None,
    img_size=IMG_SIZE,
    num_classes=NUM_CLASSES,
    patience=PATIENCE,
    min_lr=MIN_LR,
    device=DEVICE,
    use_pin_memory=USE_PIN_MEMORY,
    save_dir=SAVE_DIR,
    resume_checkpoint=None
    ):
    """
    Обучение модели U-Net для сегментации зубов с поддержкой дообучения

    Args:
        dataset_path:     путь к датасету
        learning_rate:    скорость обучения
        batch_size:       размер батча
        num_epochs:       число эпох обучения
        img_size:         размер изображения на вход модели
        num_classes:      число класов
        patience:         для борьбы с переобучением
        min_lr:           min скорость обучения
        device:           устойство
        use_pin_memory:   использование use pin memory для ускорения
        save_dir:         куда сохранять
        resume_checkpoint: путь к чекпоинту для возобновления обучения (опционально)
    """

    print(f"Используется устройство: {device}")
    if torch.cuda.is_available():
        print(f"GPU память: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

    # Проверка существования датасета
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Датасет не найден по пути: {dataset_path}")

    data_yaml_path = os.path.join(dataset_path, 'data.yaml')
    if not os.path.exists(data_yaml_path):
        raise FileNotFoundError(f"Файл конфигурации data.yaml не найден: {data_yaml_path}")

    # Трансформации для аугментации данных
    train_transform = get_train_transforms(img_size)
    val_transform = get_val_transforms(img_size)

    # Загрузка датасетов
    print("Загрузка датасетов...")
    try:
        train_dataset = TeethSegmentationDataset(
            dataset_path, split='train',
            transform=train_transform, img_size=img_size
        )

        val_dataset = TeethSegmentationDataset(
            dataset_path, split='valid',
            transform=val_transform, img_size=img_size
        )
    except Exception as e:
        raise RuntimeError(f"Ошибка при загрузке датасетов: {e}")

    # Проверка, что датасеты не пустые
    if len(train_dataset) == 0:
        raise ValueError("Тренировочный датасет пуст")
    if len(val_dataset) == 0:
        raise ValueError("Валидационный датасет пуст")

    # Адаптивное количество workers
    num_workers = min(2, len(train_dataset) // batch_size) if len(train_dataset) > batch_size else 0

    # Создание DataLoader'ов
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size,
        shuffle=True, num_workers=num_workers, pin_memory=use_pin_memory
    )

    val_loader = DataLoader(
        val_dataset, batch_size=batch_size,
        shuffle=False, num_workers=num_workers, pin_memory=use_pin_memory
    )

    print(f"Тренировочных образцов: {len(train_dataset)}")
    print(f"Валидационных образцов: {len(val_dataset)}")
    print(f"Количество workers: {num_workers}")


    # Получаем веса классов для борьбы с дисбалансом
    print("Вычисление весов классов...")
    try:
        class_weights = train_dataset.get_class_weights().to(device)
        print(f"Размер весов классов: {len(class_weights)}")

        # Проверяем корректность весов
        if len(class_weights) != num_classes:
            raise ValueError(f"Размер весов ({len(class_weights)}) не соответствует количеству классов ({num_classes})")

        print(f"Вес фона (класс 0): {class_weights[0]:.6f}")
        if len(class_weights) > 1:
            print(f"Средний вес зубов (классы 1-{len(class_weights)-1}): {class_weights[1:].mean():.4f}")
            print(f"Мин/Макс веса зубов: {class_weights[1:].min():.4f} / {class_weights[1:].max():.4f}")
        else:
            print("Внимание: получен только один класс!")
    except Exception as e:
        print(f"Ошибка при вычислении весов классов: {e}")
        print("Используем равные веса для всех классов")
        class_weights = torch.ones(num_classes).to(device)

    # Инициализация модели
    model = UNet(in_channels=1, out_channels=num_classes).to(device)

    # Функция потерь
    criterion = get_loss_function('combined', num_classes, class_weights)

    # Оптимизатор и планировщик
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )

    # Метрики (исключаем фон из средних метрик)
    class_names = train_dataset.class_names
    train_metrics = SegmentationMetrics(num_classes, class_names, exclude_background=True)
    val_metrics = SegmentationMetrics(num_classes, class_names, exclude_background=True)

    # Инициализация переменных для возобновления обучения
    start_epoch = 0
    best_val_dice = 0.0
    epochs_without_improvement = 0
    save_dir = save_dir

    # Проверяем возможность создания директории
    try:
        os.makedirs(save_dir, exist_ok=True)
        test_file = os.path.join(save_dir, 'test_write.tmp')
        with open(test_file, 'w') as f:
            f.write('test')
        os.remove(test_file)
        print(f"Модели будут сохранены в: {save_dir}")
    except Exception as e:
        print(f"Ошибка при создании директории для сохранения: {e}")
        save_dir = None
        print("Сохранение моделей отключено")

    # История обучения (включаем mAP метрики)
    training_history = {
        'epoch': [],
        'train_loss': [],
        'val_loss': [],
        'train_dice': [],
        'val_dice': [],
        'train_iou': [],
        'val_iou': [],
        'train_accuracy': [],
        'val_accuracy': [],
        'train_map50': [],
        'val_map50': [],
        'train_map50_95': [],
        'val_map50_95': [],
        'train_f1_macro': [],
        'val_f1_macro': [],
        'train_f1_micro': [],
        'val_f1_micro': [],
        'train_precision_macro': [],
        'val_precision_macro': [],
        'train_recall_macro': [],
        'val_recall_macro': [],
        'train_precision_micro': [],
        'val_precision_micro': [],
        'train_recall_micro': [],
        'val_recall_micro': [],
        'learning_rate': []
    }


    # Загрузка чекпоинта если указан
    if resume_checkpoint is not None:
        checkpoint_info = load_checkpoint(
            resume_checkpoint, model, optimizer, scheduler, device
        )

        start_epoch = checkpoint_info['start_epoch']
        best_val_dice = checkpoint_info['best_val_dice']

        # Загружаем историю обучения
        if checkpoint_info['training_history'] is not None:
            training_history = checkpoint_info['training_history']
            print(f"* История обучения загружена: {len(training_history['epoch'])} эпох")

        print(f"Целевое количество эпох: {num_epochs}")
        print(f"Осталось эпох для обучения: {num_epochs - start_epoch}")

    print("Начинаем обучение...")

    # Цикл обучения
    for epoch in range(start_epoch, num_epochs):
        # Тренировка
        model.train()
        train_metrics.reset()
        train_loss = 0.0

        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')

        for batch_idx, (images, masks) in enumerate(train_pbar):
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            # Более надежная проверка типа функции потерь
            try:
                if hasattr(criterion, 'forward') and hasattr(criterion, '__class__'):
                    result = criterion(outputs, masks)
                    if isinstance(result, tuple) and len(result) == 2:
                        loss, loss_dict = result
                    else:
                        loss = result
                        loss_dict = {'total_loss': loss.item()}
                else:
                    loss = criterion(outputs, masks)
                    loss_dict = {'total_loss': loss.item()}
            except Exception as e:
                print(f"Ошибка при вычислении функции потерь: {e}")
                raise

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_metrics.update(outputs, masks)

            train_pbar.set_postfix({
                'Loss': f"{loss.item():.4f}",
                'Avg_Loss': f"{train_loss/(batch_idx+1):.4f}"
            })

        # Валидация
        model.eval()
        val_metrics.reset()
        val_loss = 0.0

        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')

            for batch_idx, (images, masks) in enumerate(val_pbar):
                images, masks = images.to(device), masks.to(device)

                outputs = model(images)

                try:
                    if hasattr(criterion, 'forward') and hasattr(criterion, '__class__'):
                        result = criterion(outputs, masks)
                        if isinstance(result, tuple) and len(result) == 2:
                            loss, _ = result
                        else:
                            loss = result
                    else:
                        loss = criterion(outputs, masks)
                except Exception as e:
                    print(f"Ошибка при вычислении функции потерь в валидации: {e}")
                    raise

                val_loss += loss.item()
                val_metrics.update(outputs, masks)

                val_pbar.set_postfix({
                    'Loss': f"{loss.item():.4f}",
                    'Avg_Loss': f"{val_loss/(batch_idx+1):.4f}"
                })


        # Вычисляем mAP метрики в конце эпохи
        print(f"\n[Эпоха {epoch+1}] Вычисление mAP метрик...")
        print("├── Тренировочный набор...")
        train_metrics.compute_map_metrics()

        print("└── Валидационный набор...")
        val_metrics.compute_map_metrics()

        # Вычисляем остальные метрики
        try:
            train_results = train_metrics.compute()
            val_results = val_metrics.compute()
        except Exception as e:
            print(f"Ошибка при вычислении метрик: {e}")
            train_results = {'mean_dice': 0.0, 'mean_iou': 0.0, 'pixel_accuracy': 0.0,
                           'map50': 0.0, 'map50_95': 0.0, 'f1_macro': 0.0, 'f1_micro': 0.0,
                           'precision_macro': 0.0, 'recall_macro': 0.0, 'precision_micro': 0.0, 'recall_micro': 0.0}
            val_results = train_results.copy()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        # Обновляем планировщик
        scheduler.step(avg_val_loss)

        # Сохраняем историю обучения
        current_lr = optimizer.param_groups[0]['lr']

        training_history['epoch'].append(epoch + 1)
        training_history['train_loss'].append(avg_train_loss)
        training_history['val_loss'].append(avg_val_loss)
        training_history['train_dice'].append(train_results['mean_dice'])
        training_history['val_dice'].append(val_results['mean_dice'])
        training_history['train_iou'].append(train_results['mean_iou'])
        training_history['val_iou'].append(val_results['mean_iou'])
        training_history['train_accuracy'].append(train_results['pixel_accuracy'])
        training_history['val_accuracy'].append(val_results['pixel_accuracy'])
        training_history['train_map50'].append(train_results['map50'])
        training_history['val_map50'].append(val_results['map50'])
        training_history['train_map50_95'].append(train_results['map50_95'])
        training_history['val_map50_95'].append(val_results['map50_95'])
        training_history['train_f1_macro'].append(train_results['f1_macro'])
        training_history['val_f1_macro'].append(val_results['f1_macro'])
        training_history['train_f1_micro'].append(train_results['f1_micro'])
        training_history['val_f1_micro'].append(val_results['f1_micro'])
        training_history['train_precision_macro'].append(train_results['precision_macro'])
        training_history['val_precision_macro'].append(val_results['precision_macro'])
        training_history['train_recall_macro'].append(train_results['recall_macro'])
        training_history['val_recall_macro'].append(val_results['recall_macro'])
        training_history['train_precision_micro'].append(train_results['precision_micro'])
        training_history['val_precision_micro'].append(val_results['precision_micro'])
        training_history['train_recall_micro'].append(train_results['recall_micro'])
        training_history['val_recall_micro'].append(val_results['recall_micro'])
        training_history['learning_rate'].append(current_lr)

        # Выводим результаты эпохи
        print(f"\nEpoch {epoch+1}/{num_epochs}:")
        print(f"Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
        print(f"Train Dice: {train_results['mean_dice']:.4f}, Val Dice: {val_results['mean_dice']:.4f}")
        print(f"Train IoU: {train_results['mean_iou']:.4f}, Val IoU: {val_results['mean_iou']:.4f}")
        print(f"Train Acc: {train_results['pixel_accuracy']:.4f}, Val Acc: {val_results['pixel_accuracy']:.4f}")
        print(f"Train mAP@0.5: {train_results['map50']:.4f}, Val mAP@0.5: {val_results['map50']:.4f}")
        print(f"Train mAP@0.5:0.95: {train_results['map50_95']:.4f}, Val mAP@0.5:0.95: {val_results['map50_95']:.4f}")
        print(f"Train F1 (Macro): {train_results['f1_macro']:.4f}, Val F1 (Macro): {val_results['f1_macro']:.4f}")
        print(f"Train F1 (Micro): {train_results['f1_micro']:.4f}, Val F1 (Micro): {val_results['f1_micro']:.4f}")
        print(f"Train Precision (Macro): {train_results['precision_macro']:.4f}, Val Precision (Macro): {val_results['precision_macro']:.4f}")
        print(f"Train Recall (Macro): {train_results['recall_macro']:.4f}, Val Recall (Macro): {val_results['recall_macro']:.4f}")
        print(f"Learning Rate: {current_lr:.2e}")

        # Мониторинг GPU памяти
        if torch.cuda.is_available():
            gpu_memory = torch.cuda.memory_allocated() / 1024**3
            gpu_memory_max = torch.cuda.max_memory_allocated() / 1024**3
            print(f"GPU Memory: {gpu_memory:.2f}GB / Max: {gpu_memory_max:.2f}GB")


        # Early stopping и сохранение лучшей модели
        if val_results['mean_dice'] > best_val_dice:
            best_val_dice = val_results['mean_dice']
            epochs_without_improvement = 0

            # Сохраняем лучшую модель
            if save_dir is not None:
                try:
                    # Сохраняем training_history в чекпоинт
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'scheduler_state_dict': scheduler.state_dict(),
                        'best_val_dice': best_val_dice,
                        'train_results': train_results,
                        'val_results': val_results,
                        'training_history': training_history
                    }, os.path.join(save_dir, 'best_model.pth'))
                    print(f"Сохранена лучшая модель с Dice: {best_val_dice:.4f}")
                except Exception as e:
                    print(f"Ошибка при сохранении лучшей модели: {e}")
        else:
            epochs_without_improvement += 1

        # Проверка early stopping
        if epochs_without_improvement >= patience:
            print(f"Early stopping: нет улучшений {patience} эпох подряд")
            break

        # Проверка минимального learning rate
        if current_lr < min_lr:
            print(f"Learning rate ({current_lr:.2e}) меньше минимального ({min_lr:.2e}). Останавливаем обучение.")
            break

        # Сохраняем чекпоинт каждые 10 эпох
        if save_dir is not None and (epoch + 1) % 10 == 0:
            try:
                # Сохраняем training_history в чекпоинт
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'train_results': train_results,
                    'val_results': val_results,
                    'training_history': training_history
                }, os.path.join(save_dir, f'checkpoint_epoch_{epoch+1}.pth'))
                print(f"Сохранен чекпоинт эпохи {epoch+1}")
            except Exception as e:
                print(f"Ошибка при сохранении чекпоинта: {e}")

        print("-" * 50)

    ## Сохранение финальной истории и создание графиков

    # Сохраняем финальную историю обучения в JSON
    if save_dir is not None:
        try:
            history_path = os.path.join(save_dir, 'training_history.json')
            with open(history_path, 'w') as f:
                json.dump(training_history, f, indent=2)
            print(f"\n* История обучения сохранена в: {history_path}")
        except Exception as e:
            print(f"Ошибка при сохранении истории обучения: {e}")

    # Создаем графики полной истории обучения
    if save_dir is not None:
        try:
            plot_training_history(training_history, save_dir)
        except Exception as e:
            print(f"Ошибка при создании графиков: {e}")

    print("\nОбучение завершено!")
    print(f"Лучший Dice коэффициент: {best_val_dice:.4f}")
    if save_dir is not None:
        print(f"Модели сохранены в: {save_dir}")
        print(f"Графики сохранены в: {os.path.join(save_dir, 'plots')}")
    else:
        print("Модели не сохранялись из-за ошибок с директорией")

    # Очистка GPU памяти
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return training_history, save_dir

### Обучение

In [ ]:
# Этап 1: Начальное обучение
NUM_EPOCHS = 150

In [ ]:
%%time
history, save_dir = train_teeth_segmentation(
    # путь к датасету
    dataset_path=DATASET_PATH,
    # скорость обучения
    learning_rate=LEARNING_RATE,
    # размер батча
    batch_size=BATCH_SIZE,
    # число эпох обучения
    num_epochs=NUM_EPOCHS,
    # размер изображения на вход модели
    img_size=IMG_SIZE,
    # число класов (32 зуба + фон = 33)
    num_classes=NUM_CLASSES,
    # early stopping
    patience=PATIENCE,
    # min learning rate
    min_lr=MIN_LR,
    # устройство для обучения
    device=DEVICE,
    # использование use pin memory для ускорения
    use_pin_memory=USE_PIN_MEMORY,
    # каталог для сохранения результатов обучения
    save_dir=SAVE_DIR
)

### Дообучение (при необходимости)

In [ ]:
# Дообучение
# Если первичное обучение 100 эпох, а надо дообучить еще 100, то NUM_EPOCHS = 200
NUM_EPOCHS = 200

In [ ]:
%%time
history, save_dir = train_teeth_segmentation(
    # путь к датасету
    dataset_path=DATASET_PATH,
    # скорость обучения
    learning_rate=LEARNING_RATE,
    # размер батча
    batch_size=BATCH_SIZE,
    # число эпох обучения
    num_epochs=NUM_EPOCHS,
    # размер изображения на вход модели
    img_size=IMG_SIZE,
    # число класов (32 зуба + фон = 33)
    num_classes=NUM_CLASSES,
    # early stopping
    patience=PATIENCE,
    # min learning rate
    min_lr=MIN_LR,
    # устройство для обучения
    device=DEVICE,
    # использование use pin memory для ускорения
    use_pin_memory=USE_PIN_MEMORY,
    # каталог для сохранения результатов обучения
    save_dir=SAVE_DIR,

    # загрузка чекпоинта для дообучения
    resume_checkpoint=f'{save_dir}/best_model.pth'
)

### История обучения

In [ ]:
# путь для сохранения графиков истории обучения
output_dir = SAVE_DIR
# путь к файлу с историей обучения
history_path = SAVE_DIR + "/training_history.json"

In [ ]:
# Создаем папку для графиков
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

# Загружаем и визуализируем
print("Загрузка истории обучения...")
history = f.load_training_history(history_path)

f.print_training_summary(history)

print("Создание графиков...")
fig = f.plot_metrics_comparison(history, output_dir, show_plots = True)
plt.close(fig)
print(f"Графики сохранены в: {output_dir}")

## Инференс на тестовой подвыборке

In [ ]:
dataset_path = DATASET_DIR
checkpoint = SAVE_DIR + "/best_model.pth"
save_dir = SAVE_DIR
batch_size = BATCH_SIZE
img_size = IMG_SIZE
num_classes = NUM_CLASSES
num_workers = 2

In [ ]:
# Устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

# Проверка существования чекпоинта
if not os.path.exists(checkpoint):
    raise FileNotFoundError(f"Чекпоинт не найден: {checkpoint}")

# Проверка существования датасета
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Датасет не найден: {dataset_path}")

# Загружаем модель
print(f"Загрузка модели из: {checkpoint}")
model = load_model(checkpoint, num_classes, device)

# Трансформации для test набора
test_transform = get_val_transforms(img_size)

# Загружаем test датасет
print(f"Загрузка test датасета из: {dataset_path}")
test_dataset = TeethSegmentationDataset(
    dataset_path,
    split='test',
    transform=test_transform,
    img_size=img_size
)

print(f"Количество test образцов: {len(test_dataset)}")

if len(test_dataset) == 0:
    raise ValueError("Test датасет пуст!")

# Создаем DataLoader
use_pin_memory = torch.cuda.is_available()
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=use_pin_memory
)

# Получаем названия классов
class_names = test_dataset.class_names

# Запускаем инференс
results = f.run_inference(
    model,
    test_loader,
    device,
    num_classes,
    class_names,
    save_dir=save_dir
)

print("\nИнференс завершен!")
if save_dir:
    print(f"Результаты сохранены в: {save_dir}")

## Инференс на ОПТГ

In [ ]:
## Параметры

# путь к обученной модели
MODEL_PATH = SAVE_DIR + "/best_model.pth"
# путь к тестовому изображению
IMAGE_PATH = "/content/gdrive/MyDrive/Colab_Notebooks/Startup/01_Data/RealTest/№9.JPG"
# путь к data.yaml
DATA_YAML_PATH = "/content/dataset/data.yaml"
# путь к сохранению результатов инференса
OUTPUT_DIR = SAVE_DIR + "/inference_results"
# устройство для инференса
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
print(f"Используемое устройство: {DEVICE}")

# Инференс на одном изображении
mask, result_image = f.inference_single_image(
    # путь к обученной модели
    model_path=MODEL_PATH,
    # путь к тестовому изображению
    image_path=IMAGE_PATH,
    # путь к data.yaml
    data_yaml_path=DATA_YAML_PATH,
    # трансформации для тестового изображения
    transform = get_val_transforms(),
    # путь к сохранению результатов инференса
    output_dir=OUTPUT_DIR,
    # устройство для инференса
    device=DEVICE
)

In [ ]:
### Инференс на пакете изображений

## Параметры

# путь к обученной модели
MODEL_PATH = SAVE_DIR + "/best_model.pth"
# путь к пакету тестовых изображений
IMAGE_DIR = "/content/gdrive/MyDrive/Colab_Notebooks/Startup/01_Data/RealTest/"
# путь к data.yaml
DATA_YAML_PATH = "/content/dataset/data.yaml"
# путь к сохранению результатов инференса
OUTPUT_DIR = SAVE_DIR + "/inference_results"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Используемое устройство: {DEVICE}")


# Детальный инференс
masks, results = f.inference_multiple_images(
    # путь к обученной модели
    model_path=MODEL_PATH,
    # путь к пакету тестовых изображений
    image_dir=IMAGE_DIR,
    # путь к data.yaml
    data_yaml_path=DATA_YAML_PATH,
    # трансформации для пакета тестовых изображений
    transform = get_val_transforms(),
    # путь к сохранению результатов инференса
    output_dir=OUTPUT_DIR,
    # устройство для инференса
    device=DEVICE
)

## Сохранение зависимостей

In [ ]:
!pip list --format=freeze > requirements.txt